# 深層強化学習

深層強化学習は、状態空間が大きい問題で価値関数をニューラルネットで近似する方法です。表形式では持てない連続状態や高次元観測を扱える点が実務上の利点です。


## 参考動画
- [強化学習プレイリスト（外部）](https://www.youtube.com/playlist?list=PLhDAH9aTfnxI1OywfnxXCDTWGtYL2NxJR)
@[youtube](https://www.youtube.com/playlist?list=PLhDAH9aTfnxI1OywfnxXCDTWGtYL2NxJR)

## 参考リンク
- https://www.youtube.com/playlist?list=PLhDAH9aTfnxI1OywfnxXCDTWGtYL2NxJR


## 背景と目的

表形式Q学習は状態が増えるとメモリとサンプル効率の両面で限界があります。

小さなMLPでも、状態特徴から行動価値を近似できることを確認すると、DQN系の基本発想をつかめます。

ここでは NumPy だけで1隠れ層ネットを実装し、TD損失で更新します。


In [ ]:
import numpy as np
np.random.seed(0)

gamma = 0.95
lr = 0.03

# state: one-hot (s0, s1)
transitions = [
    (0, 1, 1.0, 1, False),
    (0, 0, 0.2, 0, False),
    (1, 0, 0.5, 0, False),
    (1, 1, 0.1, 1, False),
]

W1 = 0.3 * np.random.randn(2, 8)
b1 = np.zeros(8)
W2 = 0.3 * np.random.randn(8, 2)
b2 = np.zeros(2)


def one_hot(s):
    x = np.zeros(2)
    x[s] = 1.0
    return x


def forward(x):
    h_pre = x @ W1 + b1
    h = np.tanh(h_pre)
    q = h @ W2 + b2
    return h_pre, h, q


In [ ]:
for step in range(1200):
    s, a, r, s_next, done = transitions[np.random.randint(len(transitions))]
    x = one_hot(s)
    xn = one_hot(s_next)

    h_pre, h, q = forward(x)
    _, _, q_next = forward(xn)

    target = r if done else r + gamma * np.max(q_next)
    err = q[a] - target

    # backprop for selected action loss 0.5*(q[a]-target)^2
    dq = np.zeros_like(q)
    dq[a] = err

    dW2 = np.outer(h, dq)
    db2 = dq
    dh = W2 @ dq
    dh_pre = dh * (1.0 - np.tanh(h_pre) ** 2)
    dW1 = np.outer(x, dh_pre)
    db1 = dh_pre

    W2 -= lr * dW2
    b2 -= lr * db2
    W1 -= lr * dW1
    b1 -= lr * db1

    if step % 300 == 0:
        q0 = forward(one_hot(0))[2]
        q1 = forward(one_hot(1))[2]
        print(f'step={step}: Q(s0)={np.round(q0,4)}, Q(s1)={np.round(q1,4)}')


ネットワーク近似では、更新が他状態へ波及する一般化効果が得られます。ここでの実装は最小構成で、DQNで一般的な target network と experience replay は省略しています。
